# 📊 MSCI World vs Emerging Markets — Comparativa Histórica

En este notebook analizamos la evolución de los índices **MSCI World** y **MSCI Emerging Markets** desde **enero de 2009** hasta **julio de 2026**.

### Lo que haremos paso a paso:

1. 📥 **Cargar** los datos desde el archivo CSV `msci_world_em_historical.csv`
2. 📏 **Normalizar** ambos índices a **base 100** en la fecha inicial
3. 📈 **Calcular** la variación total y la rentabilidad anualizada (CAGR) de cada índice
4. 🎨 **Visualizar** la evolución con un gráfico de líneas XY (matplotlib + seaborn)

> ⚠️ Los valores absolutos no son comparables: IWRD.L cotiza en puntos del índice (~7,700) y EEM en precio por acción (~65). Por eso los normalizamos a base 100.

## 1. 📥 Carga de datos

Leemos el CSV directamente desde el repositorio de GitHub usando la URL raw. No hace falta subir el archivo manualmente a Colab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configuración de estilo
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

# Cargar datos directamente desde GitHub (raw URL)
csv_url = "https://raw.githubusercontent.com/valorar/msci/main/msci_world_em_historical.csv"
df = pd.read_csv(csv_url, parse_dates=["Date"])
df.set_index("Date", inplace=True)

print(f"Filas: {len(df):,}")
print(f"Rango: {df.index[0].strftime('%Y-%m-%d')} → {df.index[-1].strftime('%Y-%m-%d')}")
print(f"Columnas: {list(df.columns)}")
print(f"\nPrimeras 5 filas:")
df.head()

## 2. 📏 Normalización a base 100

Dividimos cada serie por su primer valor y multiplicamos por 100. Así ambos índices empiezan en 100 y podemos comparar su evolución relativa directamente.

In [ ]:
# Normalizar a base 100 (primer día = 100)
df["World100"] = df["MSCI_World"] / df["MSCI_World"].iloc[0] * 100
df["Emerging100"] = df["MSCI_EM"] / df["MSCI_EM"].iloc[0] * 100

print("✅ Columnas World100 y Emerging100 creadas")
print(f"\nPrimeras 5 filas con base 100:")
df[["MSCI_World", "World100", "MSCI_EM", "Emerging100"]].head()

## 3. 📈 Variación total y rentabilidad anualizada

### Métricas calculadas:

| Métrica | Fórmula | Interpretación |
|---------|---------|----------------|
| **Base 100 final** | (Valor final / Valor inicial) × 100 | «Tu inversión se ha multiplicado ×5.45 veces» |
| **Variación total** | (Valor final / Valor inicial) − 1 | «Ha subido un +445.35%» |
| **CAGR (anualizada)** | (Valor final / Valor inicial)^(1/años) − 1 | «Rentabilidad media anual compuesta» |

La base 100 final es la forma más intuitiva: si hubieras invertido 100 € el primer día, ¿cuánto valdrían hoy?

In [ ]:
# Valores iniciales y finales
valor_inicial_w = df["MSCI_World"].iloc[0]
valor_final_w   = df["MSCI_World"].iloc[-1]
valor_inicial_e = df["MSCI_EM"].iloc[0]
valor_final_e   = df["MSCI_EM"].iloc[-1]

# Base 100 final (multiplicador)
base100_w = valor_final_w / valor_inicial_w * 100
base100_e = valor_final_e / valor_inicial_e * 100

# Variación total (%)
var_total_w = (valor_final_w / valor_inicial_w) - 1
var_total_e = (valor_final_e / valor_inicial_e) - 1

# Años transcurridos y CAGR
dias = (df.index[-1] - df.index[0]).days
anios = dias / 365.25

cagr_w = (valor_final_w / valor_inicial_w) ** (1 / anios) - 1
cagr_e = (valor_final_e / valor_inicial_e) ** (1 / anios) - 1

print("═" * 65)
print(f"  Periodo: {df.index[0].strftime('%d/%m/%Y')} → {df.index[-1].strftime('%d/%m/%Y')}")
print(f"  Días: {dias:,}  |  Años: {anios:.2f}")
print("═" * 65)
print(f"  {'':<25} {'MSCI World':>18} {'MSCI EM':>18}")
print(f"  {'-'*61}")
print(f"  {'Valor inicial':<25} {valor_inicial_w:>18.2f} {valor_inicial_e:>18.2f}")
print(f"  {'Valor final':<25} {valor_final_w:>18.2f} {valor_final_e:>18.2f}")
print(f"  {'Base 100 final (× veces)':<25} {base100_w:>17.2f} {base100_e:>18.2f}")
print(f"  {'Variación total':<25} {var_total_w:>17.2%} {var_total_e:>18.2%}")
print(f"  {'CAGR (anualizado)':<25} {cagr_w:>17.2%} {cagr_e:>18.2%}")
print("═" * 65)
print(f"\n💡 En otras palabras:")
print(f"   MSCI World:  100 € invertidos en 2009 → {base100_w:.0f} € en 2026  (×{base100_w/100:.2f})")
print(f"   MSCI EM:     100 € invertidos en 2009 → {base100_e:.0f} € en 2026  (×{base100_e/100:.2f})")

## 4. 🎨 Gráfico de evolución (Base 100)

Visualizamos ambas series normalizadas en el mismo gráfico XY con líneas finas. Así se aprecian:

- Las **tendencias** de largo plazo de cada índice
- Los **cruces**: momentos en que uno supera al otro en rentabilidad acumulada
- La **volatilidad relativa** de emergentes frente a desarrollados

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

# Líneas finas con alpha para ver cruces
ax.plot(df.index, df["World100"], 
        color="#2563eb", linewidth=0.8, alpha=0.85, label="MSCI World")
ax.plot(df.index, df["Emerging100"], 
        color="#dc2626", linewidth=0.8, alpha=0.85, label="MSCI Emerging Markets")

# Línea horizontal en 100
ax.axhline(y=100, color="gray", linestyle="--", linewidth=0.5, alpha=0.5)

# Sombreado suave: quién va ganando en cada momento
ax.fill_between(df.index, df["World100"], df["Emerging100"],
                where=(df["World100"] >= df["Emerging100"]),
                color="#2563eb", alpha=0.06, interpolate=True)
ax.fill_between(df.index, df["World100"], df["Emerging100"],
                where=(df["World100"] < df["Emerging100"]),
                color="#dc2626", alpha=0.06, interpolate=True)

# Estilo
ax.set_title("MSCI World vs Emerging Markets — Base 100", 
             fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("")
ax.set_ylabel("Base 100", fontsize=11)
ax.legend(frameon=True, fontsize=10, loc="upper left")
ax.grid(True, alpha=0.3)

# Anotar valores finales
ultimo = df.index[-1]
ax.annotate(f"MSCI World: {df['World100'].iloc[-1]:.1f}",
            xy=(ultimo, df["World100"].iloc[-1]),
            xytext=(15, 15), textcoords="offset points",
            fontsize=9, color="#2563eb", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="#2563eb", alpha=0.6))
ax.annotate(f"MSCI EM: {df['Emerging100'].iloc[-1]:.1f}",
            xy=(ultimo, df["Emerging100"].iloc[-1]),
            xytext=(15, -15), textcoords="offset points",
            fontsize=9, color="#dc2626", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="#dc2626", alpha=0.6))

plt.tight_layout()
plt.show()

## 5. 📝 Observaciones

- El gráfico muestra los **ciclos** en que emergentes supera a desarrollados y viceversa
- Las **áreas sombreadas** (azul = World gana, rojo = EM gana) señalan quién lidera en cada momento
- La **volatilidad** de emergentes es visiblemente mayor (oscilaciones más amplias)
- La **rentabilidad anualizada (CAGR)** da una medida comparable independientemente del plazo

### Próximos análisis:

- [ ] Rebalanceo periódico entre ambos índices (50/50, 60/40, 70/30)
- [ ] Drawdowns máximos de cada índice
- [ ] Rolling returns (ventanas de 1, 3, 5 años)
- [ ] Correlación rodante